# FINAL RQ3 RESULTS — Evidence-Value Analysis with Expected Information Gain

**Purpose:** Answer RQ3 by identifying which currently unobserved clinical variable is expected to reduce diagnostic uncertainty most for held-out uncertain cases, comparing EIG with SHAP, testing ranking stability, and evaluating whether the recommended observation crosses actionable posterior thresholds.

This notebook is part of the final, unified MSc research pipeline described in the research proposal. It is designed to save intermediate artifacts to Google Drive so that long-running computations can resume after a Colab restart.

**Important:** the old processed splits and old model artifacts are not treated as final because the supervisor raised a patient-level split/data-leakage concern. The raw dataset may be reused, but the final pipeline rebuilds the split and downstream models from that raw data.

In [ ]:
# =========================
# FINAL PROJECT CONFIGURATION
# =========================
from google.colab import drive
drive.mount("/content/drive")

import os, json, pickle, shutil, hashlib, time, warnings
from pathlib import Path

PIPELINE_VERSION = "FINAL_2026-09-24_v1"
RANDOM_STATE = 42

DRIVE_ROOT = Path("/content/drive/MyDrive/heart_failure_bn_FINAL")
DIRS = {
    "root": DRIVE_ROOT,
    "checkpoints": DRIVE_ROOT / "checkpoints",
    "models": DRIVE_ROOT / "models",
    "predictions": DRIVE_ROOT / "predictions",
    "results": DRIVE_ROOT / "results",
    "figures": DRIVE_ROOT / "figures",
    "logs": DRIVE_ROOT / "logs",
    "data": DRIVE_ROOT / "data",
    "templates": DRIVE_ROOT / "templates",
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

LEGACY_ROOT = Path("/content/drive/MyDrive/heart_failure_bn_checkpoints")
print("FINAL Drive folder:", DRIVE_ROOT)
print("Pipeline version:", PIPELINE_VERSION)

# We may reuse the raw dataset only. Old processed splits/models are deliberately NOT reused.
legacy_raw = LEGACY_ROOT / "raw_master_dataset.pkl"
final_raw = DIRS["data"] / "raw_master_dataset.pkl"
if not final_raw.exists() and legacy_raw.exists():
    shutil.copy2(legacy_raw, final_raw)
    print("Copied legacy raw dataset into FINAL project folder.")
print("Raw dataset available:", final_raw.exists())


## Research alignment

The proposal defines RQ3 around uncertain heart-failure predictions, with a primary uncertainty band of 0.40–0.60 and sensitivity bands of 0.35–0.65 and 0.45–0.55. EIG is computed by averaging the KL divergence between the posterior before and after observing a candidate variable over that variable's predictive distribution. The proposal compares the resulting ranking against SHAP from the best-performing baseline classifier and evaluates ranking stability and actionability.

This notebook uses the held-out test patients for evaluation and treats each candidate variable as temporarily unobserved while keeping the remaining clinical evidence fixed.

In [ ]:
!pip install -q pgmpy shap

import os, json, pickle, math, gc, warnings
from pathlib import Path
import numpy as np
import pandas as pd

from pgmpy.inference import VariableElimination
import scipy.stats as stats
from sklearn.metrics import f1_score

warnings.filterwarnings("ignore")
np.random.seed(RANDOM_STATE)

TARGET = "heart_failure"
BASELINE_BAND = (0.40, 0.60)
SENSITIVITY_BANDS = [(0.35, 0.65), (0.45, 0.55)]
ACTION_THRESHOLDS = [0.75, 0.80, 0.85]

bn_path = DIRS["models"] / "FINAL_bayesian_network.pkl"
winners_path = DIRS["results"] / "FINAL_RQ1_winning_grids.csv"
bn_probs_path = DIRS["predictions"] / "FINAL_BN_test_prob.npy"

assert bn_path.exists(), "Final BN missing. Complete RQ1 first."
assert winners_path.exists(), "RQ1 winner table missing. Complete RQ1 first."
assert bn_probs_path.exists(), "BN test probabilities missing. Complete RQ1 first."

with open(bn_path, "rb") as f:
    bn = pickle.load(f)
winners = pd.read_csv(winners_path)

X_test_bn = pd.read_pickle(DIRS["data"] / "X_test_bn.pkl")
X_test_final = pd.read_pickle(DIRS["data"] / "X_test_final.pkl")
X_val_final = pd.read_pickle(DIRS["data"] / "X_val_final.pkl")
y_test = pd.read_pickle(DIRS["data"] / "y_test.pkl")
y_val = pd.read_pickle(DIRS["data"] / "y_val.pkl")

for df_ in [X_test_bn]:
    for c in df_.columns:
        df_[c] = df_[c].astype(str)

bn_test_probs = np.load(bn_probs_path)
ve = VariableElimination(bn)
candidate_vars = [v for v in bn.nodes() if v != TARGET]

print("BN nodes:", len(bn.nodes()), "edges:", len(bn.edges()))
print("Test cases:", len(X_test_bn))
print("Uncertain 0.40–0.60:", int(((bn_test_probs >= .40) & (bn_test_probs <= .60)).sum()))


## 1. Select the baseline classifier using validation-set F1

In [ ]:
# The proposal specifies selection using validation-set F1.
best_baseline = None
best_f1 = -np.inf
best_model_path = None

for _, row in winners.iterrows():
    model_name = row["model"]
    grid_name = row["grid"]
    model_path = DIRS["models"] / f"{model_name}__{grid_name}.pkl"
    with open(model_path, "rb") as f:
        model = pickle.load(f)

    val_prob = model.predict_proba(X_val_final)[:, 1]
    val_pred = (val_prob >= 0.50).astype(int)
    val_f1 = f1_score(y_val, val_pred, zero_division=0)

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_baseline = model_name
        best_model_path = model_path

print("Best baseline by validation F1:", best_baseline)
print("Validation F1:", best_f1)
print("Model path:", best_model_path)

with open(best_model_path, "rb") as f:
    best_baseline_model = pickle.load(f)

pd.DataFrame([{
    "selected_baseline": best_baseline,
    "validation_F1": best_f1
}]).to_csv(DIRS["results"] / "FINAL_RQ3_selected_SHAP_baseline.csv", index=False)


## 2. Identify uncertain test cases

In [ ]:
uncertain_masks = {}
for lo, hi in [BASELINE_BAND] + SENSITIVITY_BANDS:
    key = f"{lo:.2f}_{hi:.2f}"
    mask = (bn_test_probs >= lo) & (bn_test_probs <= hi)
    uncertain_masks[key] = mask
    print(key, int(mask.sum()))

# We calculate EIG once for the widest band and reuse cases for narrower bands.
widest_lo, widest_hi = 0.35, 0.65
widest_mask = uncertain_masks["0.35_0.65"]
case_indices = np.where(widest_mask)[0]

case_index_path = DIRS["results"] / "FINAL_RQ3_uncertain_case_indices.csv"
pd.DataFrame({
    "test_position": case_indices,
    "bn_p_hf": bn_test_probs[case_indices]
}).to_csv(case_index_path, index=False)

## 3. EIG computation with resumable per-case checkpointing

In [ ]:
def binary_entropy(p):
    p = float(np.clip(p, 1e-12, 1 - 1e-12))
    return -(p*np.log(p) + (1-p)*np.log(1-p))

def joint_to_eig(q, target_name, candidate_name):
    # q is a factor over [candidate, target] (ordering can vary).
    states_x = [str(s) for s in q.state_names[candidate_name]]
    states_d = [str(s) for s in q.state_names[target_name]]
    joint = np.asarray(q.values, dtype=float)

    # Reorder axes robustly to candidate x target.
    vars_q = list(q.variables)
    x_axis = vars_q.index(candidate_name)
    d_axis = vars_q.index(target_name)
    if (x_axis, d_axis) != (0, 1):
        joint = np.moveaxis(joint, [x_axis, d_axis], [0, 1])

    p_x = joint.sum(axis=1)
    p_d = joint.sum(axis=0)

    target_states = states_d
    if "1" in target_states:
        p1 = float(p_d[target_states.index("1")])
    else:
        p1 = float(p_d[-1])

    h_before = binary_entropy(p1)

    expected_h_after = 0.0
    for i, px in enumerate(p_x):
        if px <= 0:
            continue
        p_d_given_x = joint[i] / px
        if "1" in target_states:
            px_p1 = float(p_d_given_x[target_states.index("1")])
        else:
            px_p1 = float(p_d_given_x[-1])
        expected_h_after += float(px) * binary_entropy(px_p1)

    return max(0.0, h_before - expected_h_after), p1, p_x, states_x

eig_pickle = DIRS["checkpoints"] / "FINAL_RQ3_eig_case_results.pkl"

if eig_pickle.exists():
    with open(eig_pickle, "rb") as f:
        eig_records = pickle.load(f)
    completed_cases = set(r["test_position"] for r in eig_records)
    print("Resuming EIG cases:", len(completed_cases), "/", len(case_indices))
else:
    eig_records = []
    completed_cases = set()

for n_done, pos in enumerate(case_indices, start=1):
    if int(pos) in completed_cases:
        continue

    row = X_test_bn.iloc[pos]
    # Current diagnostic evidence = every observed clinical variable except target.
    full_evidence = {v: str(row[v]) for v in candidate_vars}

    record = {
        "test_position": int(pos),
        "p_hf_full": float(bn_test_probs[pos]),
    }

    # The EIG query for candidate X removes X from the evidence, then considers all values of X.
    for var in candidate_vars:
        evidence = {k: v for k, v in full_evidence.items() if k != var}
        q = ve.query([var, TARGET], evidence=evidence, show_progress=False)
        eig, p_before, p_x, states_x = joint_to_eig(q, TARGET, var)
        record[f"EIG__{var}"] = float(eig)
        record[f"P_before__{var}"] = float(p_before)

        # Actual held-out value, used only for evaluation of actionability.
        actual_x = str(row[var])
        if actual_x in states_x:
            ix = states_x.index(actual_x)
            actual_joint = np.asarray(q.values, dtype=float)
            vars_q = list(q.variables)
            x_axis = vars_q.index(var); d_axis = vars_q.index(TARGET)
            if (x_axis, d_axis) != (0,1):
                actual_joint = np.moveaxis(actual_joint, [x_axis, d_axis], [0,1])
            px = float(actual_joint[ix].sum())
            if px > 0:
                d_given_actual = actual_joint[ix] / px
                target_states = [str(s) for s in q.state_names[TARGET]]
                p_after = float(d_given_actual[target_states.index("1")]) if "1" in target_states else float(d_given_actual[-1])
            else:
                p_after = np.nan
        else:
            p_after = np.nan
        record[f"P_after_actual__{var}"] = p_after

    eig_records.append(record)

    if n_done % 10 == 0 or n_done == 1:
        with open(eig_pickle, "wb") as f:
            pickle.dump(eig_records, f)
        pd.DataFrame(eig_records).to_csv(DIRS["results"] / "FINAL_RQ3_EIG_per_case.csv", index=False)
        print(f"Checkpointed {len(eig_records)}/{len(case_indices)} cases")

with open(eig_pickle, "wb") as f:
    pickle.dump(eig_records, f)
eig_df = pd.DataFrame(eig_records)
eig_df.to_csv(DIRS["results"] / "FINAL_RQ3_EIG_per_case.csv", index=False)
print("EIG complete:", eig_df.shape)



## 4. Aggregate EIG rankings for all uncertainty bands

In [ ]:
ranking_tables = {}

for lo, hi in [BASELINE_BAND] + SENSITIVITY_BANDS:
    subset = eig_df[
        (eig_df["p_hf_full"] >= lo) &
        (eig_df["p_hf_full"] <= hi)
    ].copy()

    rows = []
    for var in candidate_vars:
        col = f"EIG__{var}"
        if col not in subset.columns:
            continue
        rows.append({
            "band_low": lo,
            "band_high": hi,
            "variable": var,
            "n_cases": len(subset),
            "mean_EIG": float(subset[col].mean()),
            "median_EIG": float(subset[col].median()),
        })

    band_df = pd.DataFrame(rows).sort_values("mean_EIG", ascending=False).reset_index(drop=True)
    band_df["EIG_rank"] = np.arange(1, len(band_df)+1)

    key = f"{lo:.2f}_{hi:.2f}"
    ranking_tables[key] = band_df
    band_df.to_csv(DIRS["results"] / f"FINAL_RQ3_EIG_ranking_{key}.csv", index=False)

print(ranking_tables["0.40_0.60"].head(20).to_string(index=False))

## 5. SHAP ranking from the best baseline classifier

In [ ]:
import shap

# Use a manageable reference/background sample for explainers.
background = X_val_final.sample(min(300, len(X_val_final)), random_state=RANDOM_STATE)
explain_X = X_test_final.sample(min(2000, len(X_test_final)), random_state=RANDOM_STATE)

shap_summary_path = DIRS["results"] / "FINAL_RQ3_SHAP_variable_importance.csv"

if shap_summary_path.exists():
    shap_importance_df = pd.read_csv(shap_summary_path)
else:
    try:
        explainer = shap.Explainer(best_baseline_model, background)
        shap_values = explainer(explain_X)
        values = shap_values.values
    except Exception:
        # Fallback for models whose automatic explainer is unavailable.
        explainer = shap.PermutationExplainer(best_baseline_model.predict_proba, background.iloc[:100])
        shap_values = explainer(explain_X.iloc[:300])
        values = shap_values.values

    values = np.asarray(values)
    if values.ndim == 3:
        # binary classification -> positive class
        values = values[:, :, 1]

    feature_names = list(explain_X.columns)
    mean_abs = np.abs(values).mean(axis=0)

    # Map one-hot classifier columns back to their underlying clinical variable.
    bn_nodes = set(candidate_vars)
    def map_feature(f):
        if f in bn_nodes:
            return f
        if "_" in f:
            prefix = f.split("_")[0]
        for var in bn_nodes:
            if f.startswith(var + "_"):
                return var
        return f

    rows = [{"classifier_feature": f, "bn_variable": map_feature(f), "mean_abs_shap": float(s)}
            for f, s in zip(feature_names, mean_abs)]
    shap_features = pd.DataFrame(rows)

    shap_importance_df = (
        shap_features.groupby("bn_variable")["mean_abs_shap"]
        .sum()
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns={"bn_variable": "variable"})
    )
    shap_importance_df["SHAP_rank"] = np.arange(1, len(shap_importance_df)+1)
    shap_importance_df.to_csv(shap_summary_path, index=False)

print("SHAP baseline:", best_baseline)
print(shap_importance_df.head(20).to_string(index=False))


## 6. EIG vs SHAP rank correlations

In [ ]:
correlation_rows = []

for key, band_df in ranking_tables.items():
    merged = band_df[["variable", "EIG_rank"]].merge(
        shap_importance_df[["variable", "SHAP_rank"]],
        on="variable", how="inner"
    )

    if len(merged) >= 3:
        rho, p = stats.spearmanr(merged["EIG_rank"], merged["SHAP_rank"])
    else:
        rho, p = np.nan, np.nan

    correlation_rows.append({
        "band": key,
        "n_shared_variables": len(merged),
        "spearman_rho_EIG_vs_SHAP": rho,
        "p_value": p
    })

correlation_df = pd.DataFrame(correlation_rows)
correlation_df.to_csv(DIRS["results"] / "FINAL_RQ3_EIG_vs_SHAP_correlations.csv", index=False)
print(correlation_df.to_string(index=False))


## 7. Ranking stability across uncertainty bands

In [ ]:
baseline_key = "0.40_0.60"
base_rank = ranking_tables[baseline_key][["variable", "EIG_rank"]]

stability_rows = []
for key, table in ranking_tables.items():
    if key == baseline_key:
        continue
    merged = base_rank.merge(table[["variable", "EIG_rank"]], on="variable", suffixes=("_base", "_alt"))
    if len(merged) >= 3:
        rho, p = stats.spearmanr(merged["EIG_rank_base"], merged["EIG_rank_alt"])
    else:
        rho, p = np.nan, np.nan
    stability_rows.append({
        "baseline_band": baseline_key,
        "alternative_band": key,
        "n_shared_variables": len(merged),
        "spearman_rho": rho,
        "p_value": p
    })

stability_df = pd.DataFrame(stability_rows)
stability_df.to_csv(DIRS["results"] / "FINAL_RQ3_EIG_band_stability.csv", index=False)
print(stability_df.to_string(index=False))

## 8. Actionability evaluation at 0.75 / 0.80 / 0.85

In [ ]:
# Use the top EIG variable from the primary 0.40–0.60 band for each case.
primary_subset = eig_df[
    (eig_df["p_hf_full"] >= BASELINE_BAND[0]) &
    (eig_df["p_hf_full"] <= BASELINE_BAND[1])
].copy()

primary_rank = ranking_tables["0.40_0.60"]["variable"].tolist()

action_rows = []
for _, rec in primary_subset.iterrows():
    pos = int(rec["test_position"])
    row = X_test_bn.iloc[pos]

    # Select highest-EIG variable for THIS case.
    per_case = []
    for var in candidate_vars:
        per_case.append((var, rec.get(f"EIG__{var}", np.nan)))
    per_case = [(v,e) for v,e in per_case if np.isfinite(e)]
    if not per_case:
        continue
    top_var, top_eig = max(per_case, key=lambda x: x[1])

    p_before = float(rec[f"P_before__{top_var}"])
    p_after = float(rec[f"P_after_actual__{top_var}"])

    action_rows.append({
        "test_position": pos,
        "top_eig_variable": top_var,
        "top_eig": top_eig,
        "p_before": p_before,
        "p_after_actual": p_after,
        "actual_state": str(row[top_var])
    })

action_df = pd.DataFrame(action_rows)

action_summary = []
for threshold in ACTION_THRESHOLDS:
    crossed = (
        (action_df["p_before"] < threshold) &
        (action_df["p_after_actual"] >= threshold)
    )
    action_summary.append({
        "threshold": threshold,
        "n_uncertain_cases": len(action_df),
        "n_crossed": int(crossed.sum()),
        "proportion_crossed": float(crossed.mean()) if len(crossed) else np.nan
    })

action_df.to_csv(DIRS["results"] / "FINAL_RQ3_actionability_per_case.csv", index=False)
action_summary_df = pd.DataFrame(action_summary)
action_summary_df.to_csv(DIRS["results"] / "FINAL_RQ3_actionability_summary.csv", index=False)

print(action_summary_df.to_string(index=False))


## 9. Final RQ3 summary

In [ ]:
primary = ranking_tables["0.40_0.60"]
top_variable = primary.iloc[0]["variable"] if len(primary) else None

summary = pd.DataFrame([{
    "selected_SHAP_baseline": best_baseline,
    "validation_F1": best_f1,
    "n_cases_primary_band_0.40_0.60": int((
        (bn_test_probs >= 0.40) & (bn_test_probs <= 0.60)
    ).sum()),
    "top_EIG_variable_primary_band": top_variable,
    "top_EIG_value_primary_band": float(primary.iloc[0]["mean_EIG"]) if len(primary) else np.nan,
}])

summary.to_csv(DIRS["results"] / "FINAL_RQ3_RESULTS.csv", index=False)
print(summary.to_string(index=False))

## Resume / checkpoint behavior

The expensive RQ3 EIG stage writes `FINAL_RQ3_eig_case_results.pkl` and `FINAL_RQ3_EIG_per_case.csv` into the Drive checkpoint folder after every batch of cases. Restarting the Colab session and rerunning this notebook resumes from the already completed test positions.

All final RQ3 tables are written under:

`MyDrive/heart_failure_bn_FINAL/results/`